# RAG Day 4

## Evaluation!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Keep in mind how you would evaluate RAG for your business</h2>
            <span style="color:#181;">This is such an important part of building an accurate and reliable RAG pipeline. And it's applicable to many aspects of solving business problems with LLMs. People are often focused on RAG architecture and RAG frameworks for their business. But even more important: evaluations!</span>
        </td>
    </tr>
</table>

# Evaluation Metrics for Ranking & Retrieval

---

## 1. Precision@K
**"Of the top K results I showed, how many were actually relevant?"**

You search for "action movies" and the system returns 5 results. 3 of them are actually action movies.

**Precision@5 = 3/5 = 0.6**

It doesn't care *where* in the list the good results appear — just how many of the K results were correct.

---

## 2. Recall@K
**"Of all the relevant items that exist, how many did I find in my top K results?"**

There are 10 action movies in the entire database. Your system returned 5 results, and 3 of them were action movies.

**Recall@5 = 3/10 = 0.3**

It penalizes you for missing relevant items that existed but weren't surfaced.

---

## 3. Mean Reciprocal Rank (MRR)
**"How quickly did the first correct answer appear?"**

You ask a question and the system returns a ranked list. You only care about where the *first* relevant result lands.

| Query | First Correct Result at Rank | Reciprocal Rank |
|---|---|---|
| Q1 | Rank 1 | 1/1 = 1.0 |
| Q2 | Rank 3 | 1/3 ≈ 0.33 |
| Q3 | Rank 2 | 1/2 = 0.5 |

**MRR = (1.0 + 0.33 + 0.5) / 3 ≈ 0.61**

A score of 1.0 means the correct answer was always first. Lower scores mean you're making users scroll to find it.

---

## 4. nDCG (Normalized Discounted Cumulative Gain)
**"Did I put the *most* relevant results at the *top*?"**

This is the richest metric. It accounts for the fact that position matters — rank 1 is far more valuable than rank 5.

Say you have 4 results with relevance scores (3 = very relevant, 1 = somewhat, 0 = not relevant):

- **Your system returned:** [3, 0, 1, 3] — good result buried at rank 4
- **Ideal order would be:** [3, 3, 1, 0] — best results first

nDCG computes your system's score as a fraction of the ideal score. The "discount" means results at lower ranks contribute less, and "normalized" means you're always compared against the perfect ranking.

**nDCG = your DCG / ideal DCG** — a value between 0 and 1, where 1.0 is perfect ordering.

---

## Quick Comparison

| Metric | Cares about order? | Cares about missing items? | Best for |
|---|---|---|---|
| Precision@K | No | No | "Are my results clean?" |
| Recall@K | No | Yes | "Am I finding everything?" |
| MRR | Yes (first hit only) | No | Q&A, single correct answer |
| nDCG | Yes (all positions) | Partial | Recommendations, search ranking |

> For a recommender system (e.g., MovieLens), **nDCG and Recall@K** are the most commonly reported, since you care both about covering relevant items and surfacing the best ones near the top.

In [1]:
from evaluation import test

In [2]:
tests = test.load_tests()

In [3]:
len(tests)

150

In [4]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [5]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [6]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [7]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.16666666666666666, ndcg=0.28711770538226206, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [11]:
eval, answer, chunks = evaluate_answer(example)

In [12]:
eval

AnswerEval(feedback="The answer correctly identifies Maxine as the winner and mentions the IIOTY award in 2023, but it omits Thompson's full name, which is present in the reference. This affects completeness. The relevance is high, as it directly addresses the question about the award winner.", accuracy=5.0, completeness=4.0, relevance=5.0)

In [13]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The answer correctly identifies Maxine as the winner and mentions the IIOTY award in 2023, but it omits Thompson's full name, which is present in the reference. This affects completeness. The relevance is high, as it directly addresses the question about the award winner.
5.0
4.0
5.0
